<h2>Import Libraries</h2>

In [6]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

<h2>Download NLTK Data</h2>

In [7]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\siddharth\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

<h2>Load Dataset</h2>

In [8]:
# Example: IMDb dataset CSV (change path if needed)
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


<h2>Data Understanding</h2>

In [9]:
print("Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

print("\nSample Data:")
print(df.head())

Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Data:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


<h2>Text Preprocessing Funtions</h2>

In [10]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords + stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    
    return " ".join(words)

<h2>Apply Preprocessing</h2>

In [11]:
df['clean_text'] = df['review'].apply(clean_text)

df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


<h2>Train-Test_Split</h2>

In [12]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

<h2>Feature Engineering (BoW)</h2>

In [13]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

<h2>Feature Engineering (TF-IDF)</h2>

In [14]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

<h2>Model Training Function</h2>

In [15]:
def train_evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds, pos_label='positive'))
    print("Recall:", recall_score(y_test, preds, pos_label='positive'))
    print("F1 Score:", f1_score(y_test, preds, pos_label='positive'))
    print("\nClassification Report:\n", classification_report(y_test, preds))
    
    return accuracy_score(y_test, preds)

<h2>Logistic Regression</h2>

In [16]:
print("Logistic Regression (TF-IDF)")
lr = LogisticRegression()
lr_acc = train_evaluate(lr, X_train_tfidf, X_test_tfidf, y_train, y_test)

Logistic Regression (TF-IDF)
Accuracy: 0.8846
Precision: 0.8739172281039461
Recall: 0.9009724151617384
F1 Score: 0.8872386163767833

Classification Report:
               precision    recall  f1-score   support

    negative       0.90      0.87      0.88      4961
    positive       0.87      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.89      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



<h2>Naive Bayes</h2>

In [17]:
print("Naive Bayes (TF-IDF)")
nb = MultinomialNB()
nb_acc = train_evaluate(nb, X_train_tfidf, X_test_tfidf, y_train, y_test)

Naive Bayes (TF-IDF)
Accuracy: 0.8467
Precision: 0.8441303494307028
Recall: 0.8533439174439373
F1 Score: 0.8487121286884437

Classification Report:
               precision    recall  f1-score   support

    negative       0.85      0.84      0.84      4961
    positive       0.84      0.85      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



<h2>Decision Tree</h2>

In [18]:
print("Decision Tree (TF-IDF)")
dt = DecisionTreeClassifier()
dt_acc = train_evaluate(dt, X_train_tfidf, X_test_tfidf, y_train, y_test)

Decision Tree (TF-IDF)
Accuracy: 0.7146
Precision: 0.7180203552185193
Recall: 0.714030561619369
F1 Score: 0.7160199004975124

Classification Report:
               precision    recall  f1-score   support

    negative       0.71      0.72      0.71      4961
    positive       0.72      0.71      0.72      5039

    accuracy                           0.71     10000
   macro avg       0.71      0.71      0.71     10000
weighted avg       0.71      0.71      0.71     10000



<h2>Comparison Table</h2>

In [24]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [lr_acc, nb_acc, dt_acc]
})

results

,Model,Accuracy
0,Logistic Regression,0.8846
1,Naive Bayes,0.8467
2,Decision Tree,0.7146


<h2>Key Insights</h2>

Insights:

1. Preprocessing Impact:
   - Removing stopwords + stemming reduced noise
   - Helped improve model performance

2. Vectorization:
   - TF-IDF performed better than BoW (captures importance)

3. Model Comparison:
   - Logistic Regression → Best overall performance
   - Naive Bayes → Fast and decent accuracy
   - Decision Tree → Lower performance, overfitting tendency

4. Trade-offs:
   - Logistic Regression: Accurate but slightly slower
   - Naive Bayes: Fast but less powerful
   - Decision Tree: Easy to interpret but unstable

Conclusion:
TF-IDF + Logistic Regression is the best combination for this dataset.

<h2>Prediction Demo</h2>

In [27]:
def predict_sentiment(text):
    # Preprocess
    text_clean = clean_text(text)
    
    # Vectorize
    text_vec = tfidf.transform([text_clean])
    
    # Predict
    pred = lr.predict(text_vec)[0]
    
    return pred

# --- Predefined test examples ---
tests = [
    "This movie was fantastic!",
    "Worst film I have ever seen",
    "It was okay, not great but not bad",
    "Absolutely loved the acting and story",
    "Terrible experience, waste of time"
]

print("----- Sample Predictions -----\n")
for t in tests:
    print("Text:", t)
    print("Prediction:", predict_sentiment(t))
    print("-"*50)

----- Sample Predictions -----

Text: This movie was fantastic!
Prediction: positive
--------------------------------------------------
Text: Worst film I have ever seen
Prediction: negative
--------------------------------------------------
Text: It was okay, not great but not bad
Prediction: negative
--------------------------------------------------
Text: Absolutely loved the acting and story
Prediction: positive
--------------------------------------------------
Text: Terrible experience, waste of time
Prediction: negative
--------------------------------------------------
